# 33 · Graph store — Neo4j: relationships as first-class

**Neo4j is the mesh's graph store.** Where every other store in the query wave keeps *rows*,
*documents*, or *columns* and treats the links between them as foreign keys you `JOIN` back
together at read time, Neo4j stores the **relationships themselves** as first-class, on-disk
objects. A node points *directly* at its neighbours, so walking from one entity to a related
one is a pointer hop, not a join — and a query that would be a pile of self-joins in SQL is a
short **pattern** in Cypher.

That is the whole reason a graph store earns a place in the mesh:

> **Store the relationships, not just the rows. Walk them by pattern, not by join.
> The questions that are cheap here are the ones that are expensive everywhere else:
> multi-hop traversals, "things connected to things connected to things".**

This notebook connects to the live Neo4j, **discovers the real graph model** (it does not
assume one), then asks the graph three questions it is uniquely good at: a **multi-hop
traversal**, a **co-occurrence pattern match**, and a **degree-centrality aggregation** — the
native-Cypher shape of a classic graph algorithm.

> **Read-only, throughout.** Every query here is `MATCH ... RETURN`. Nothing is created,
> merged, or deleted — no `CREATE` / `MERGE` / `DELETE`, and (see the GDS section) no
> persistent graph projection is ever left behind. Because we write nothing, there is **no
> cleanup section**.

## Setup

The `neo4j` Python driver is **not** in the singleuser base image (which ships `polars`,
`s3fs`, `pyarrow`, `duckdb`, `fastavro`), so we install it here. `polars` — used to render
every result frame, exactly as in notebooks `20` / `22` — already ships in the image.

In [1]:
%pip install -q neo4j


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: /tmp/nb33venv/bin/python3 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**. The committed default is the **in-cluster** Bolt URL
(`neo4j://neo4j.weyland.svc.cluster.local:7687`); a validation run overrides `NEO4J_URI`
(e.g. to the store's NodePort) **without editing the notebook** — the same pattern the other
query-wave notebooks use. Neo4j is **unmeshed** and **requires auth**: user `neo4j`, password
supplied only through `NEO4J_PASSWORD` (never committed).

`driver.verify_connectivity()` proves the connection by opening a real Bolt session — we do
not echo the resolved address. `run(cypher)` is our tiny helper: execute a read query and hand
the rows back as a list of dicts; `df(rows)` renders them as a **polars** DataFrame (house
style, as in `20` / `22`).

In [2]:
import os
import polars as pl
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ.get("NEO4J_URI", "neo4j://neo4j.weyland.svc.cluster.local:7687"),
    auth=(os.environ.get("NEO4J_USER", "neo4j"), os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()   # proves the connection, not the address

# run(cypher) -> execute a read query, return rows as a list of dicts
# df(rows)    -> render a list-of-dicts as a polars DataFrame (house style, as in 20/22)
def run(cypher, **params):
    with driver.session() as s:
        return [r.data() for r in s.run(cypher, **params)]

def df(rows):
    return pl.DataFrame(rows) if rows else pl.DataFrame()

comp = run("CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition")[0]
print("Neo4j    :", comp["name"], comp["versions"][0], f"({comp['edition']} edition)")
print("as user  :", os.environ.get("NEO4J_USER", "neo4j"))

Neo4j    : Neo4j Kernel 5.26.28 (community edition)
as user  : neo4j


## Discover — what is actually in this graph?

Neo4j is self-describing. `db.labels()` lists the **node labels** (the kinds of things),
`db.relationshipTypes()` lists the **relationship types** (the kinds of links), and APOC's
`apoc.meta.stats()` gives the population — how many nodes carry each label and how many edges
of each type exist. **We discover the model before writing a single traversal**, because the
right pattern depends entirely on what labels and relationships really exist here.

First, the node labels and their counts.

In [3]:
labels = [r["label"] for r in run("CALL db.labels() YIELD label RETURN label ORDER BY label")]
stats  = run("CALL apoc.meta.stats() YIELD nodeCount, relCount, labels, relTypes "
             "RETURN nodeCount, relCount, labels, relTypes")[0]

label_rows = [{"label": l, "nodes": stats["labels"].get(l, 0)} for l in labels]
label_rows.sort(key=lambda r: r["nodes"], reverse=True)
print(f"{stats['nodeCount']:,} nodes total across {len(labels)} labels")
df(label_rows)

634,813 nodes total across 14 labels


label,nodes
str,i64
"""User""",287083
"""Artist""",173992
"""Track""",109727
"""Clip""",35824
"""Album""",15713
…,…
"""Tag""",322
"""Genre""",164
"""Vertical""",11


Two domains share this one graph, which the discovery makes plain:

- a **music-listening** domain — `User`, `Artist`, `Track`, `Album`, `Genre`, `Label` (the
  FMA / Last.fm world, with tens of millions of play edges), and
- an **AI-DLC knowledge** domain — `Document`, `Chunk`, `Stage`, `Vertical`, `Tag`, `Entry`
  (workflow docs chunked for retrieval, linked to the lifecycle stage they surface at).

Now the relationship types and their populations — the *edges* are where a graph store spends
its value. Note `PLAYS` dominates by orders of magnitude: this is a real interaction graph.

In [4]:
rel_rows = [{"relationship": k.strip("()").split("[:")[1].rstrip("]->() "), "edges": v}
            for k, v in stats["relTypes"].items()]
# apoc returns several projections of each type; keep the bare "()-[:REL]->()" totals, dedup
seen, clean = set(), []
for r in sorted(rel_rows, key=lambda x: x["edges"], reverse=True):
    if r["relationship"] not in seen:
        seen.add(r["relationship"]); clean.append(r)
print(f"{stats['relCount']:,} relationships total across {len(clean)} types")
df(clean)

14,328,841 relationships total across 24 types


relationship,edges
str,i64
"""PLAYS""",13850093
"""PLAYS]->(:Artist""",13850093
"""IN_GENRE""",260586
"""IN_GENRE]->(:Genre""",260586
"""ON]->(:Album""",108686
…,…
"""SURFACES_AT]->(:Stage""",1082
"""SUBGENRE_OF""",146
"""SUBGENRE_OF]->(:Genre""",146


The **property keys** in the store — the attributes hanging off nodes and edges. Discovering
these is what lets the traversals below reference *real* properties (`genre_title`, not a
guessed `name`; `track_listens`, `play_count`) instead of assuming a schema.

In [5]:
keys = [r["propertyKey"] for r in
        run("CALL db.propertyKeys() YIELD propertyKey RETURN propertyKey ORDER BY propertyKey")]
df([{"property_key": k} for k in keys])

property_key
str
"""age"""
"""album_id"""
"""album_title"""
"""chunk_index"""
"""chunk_title"""
…
"""user_id"""
"""uuid"""
"""version"""


### The shape we found, in one picture

Discovery (above) resolves the model without any assumption. The relationships that matter for
what follows:

```
(:User)-[:PLAYS]->(:Artist)              -- the interaction graph (~13.9M edges)
(:Track)-[:ON]->(:Album)                 -- track belongs to album
(:Track)-[:IN_GENRE]->(:Genre)           -- track tagged with genre
(:Genre)-[:SUBGENRE_OF]->(:Genre)        -- genre hierarchy (self-referential)
(:Chunk)-[:BELONGS_TO]->(:Document)      -- retrieval chunk of a doc
(:Document)-[:SURFACES_AT]->(:Stage)     -- doc belongs to an AI-DLC stage
```

Key property gotchas discovery surfaced, which the queries below respect: `Genre` uses
`genre_title` (not `name`), `Track` uses `track_title` / `track_listens`, `Artist` carries only
`name`, and `User` carries `country` / `age` / `gender`. Assuming `name` on a `Genre` would have
returned all-null.

## Traversal 1 — a multi-hop pattern the graph makes trivial

Here is the first thing a graph store does that a row store does not: **walk a self-referential
hierarchy and aggregate across it in one pattern.** A `Track` is tagged with a sub-`Genre`
(`IN_GENRE`), and that sub-genre rolls up to a parent `Genre` (`SUBGENRE_OF`). We match the full
two-hop path `Track -> subgenre -> parent genre` and count the tracks that land under each
sub-genre — a relational engine would need a self-join on the genre table plus a join to tracks;
in Cypher it is one line of pattern.

In [6]:
df(run("""
    MATCH (t:Track)-[:IN_GENRE]->(sub:Genre)-[:SUBGENRE_OF]->(parent:Genre)
    RETURN parent.genre_title AS parent_genre,
           sub.genre_title    AS subgenre,
           count(DISTINCT t)  AS n_tracks
    ORDER BY n_tracks DESC
    LIMIT 10
"""))

parent_genre,subgenre,n_tracks
str,str,i64
"""Experimental""","""Avant-Garde""",9183
"""Pop""","""Experimental Pop""",7330
"""Experimental""","""Noise""",7285
"""Instrumental""","""Ambient""",7266
"""Experimental""","""Electroacoustic""",6133
"""Rock""","""Lo-Fi""",6075
"""Rock""","""Indie-Rock""",5756
"""Electronic""","""Ambient Electronic""",5747
"""Instrumental""","""Soundtrack""",5665


Read that as a rollup *through* the genre hierarchy: each row is a sub-genre, the parent it
hangs under, and how many tracks carry it. The `IN_GENRE` then `SUBGENRE_OF` chain is the
traversal — two pointer hops from a track to the top of its genre tree.

## Traversal 2 — the pattern that is *expensive everywhere else*

This is the canonical graph query: **"users who played X also played Y".** It is a two-hop
walk out from a seed artist, *back through the listeners*, and out again to everything else
those listeners played:

```
(seed:Artist) <-[:PLAYS]- (u:User) -[:PLAYS]-> (other:Artist)
```

In a relational store this is a self-join of a 13.9-million-row play table against itself —
the query every SQL "customers who bought this also bought" recommender dreads. In a graph store
it is a pattern match anchored on one artist, and it walks only that artist's listeners, so it
stays cheap. We seed on `radiohead` and rank the co-listened artists by how many listeners they
share.

In [7]:
df(run("""
    MATCH (seed:Artist)<-[:PLAYS]-(u:User)-[:PLAYS]->(other:Artist)
    WHERE seed.name = $artist AND other <> seed
    RETURN other.name          AS also_played,
           count(DISTINCT u)    AS shared_listeners
    ORDER BY shared_listeners DESC
    LIMIT 10
""", artist="radiohead"))

also_played,shared_listeners
str,i64
"""the beatles""",23149
"""coldplay""",20948
"""muse""",16858
"""pink floyd""",13136
"""sigur rós""",12923
"""red hot chili peppers""",12216
"""beck""",11882
"""the killers""",11821
"""nirvana""",11453


Every row is a real collaborative-filtering signal computed live off the interaction graph:
these are the artists most co-listened with the seed, ranked by shared audience — no
pre-materialised similarity table, no ETL, just a walk through the `PLAYS` edges.

## Aggregation — degree centrality in native Cypher

A graph aggregation counts a node's edges to score its importance. **Degree centrality** — how
many distinct listeners an artist has — is the simplest such measure, and it is just a count of
in-`PLAYS` edges grouped by artist. This touches the whole ~13.9M-edge interaction graph, so it
is the heaviest query in the notebook (a few seconds); that it runs at all in one line is the
graph store's point.

In [8]:
df(run("""
    MATCH (a:Artist)<-[:PLAYS]-(u:User)
    RETURN a.name           AS artist,
           count(DISTINCT u) AS listeners
    ORDER BY listeners DESC
    LIMIT 10
"""))

artist,listeners
str,i64
"""radiohead""",61984
"""the beatles""",61010
"""coldplay""",53400
"""red hot chili peppers""",39135
"""muse""",37509
"""metallica""",36115
"""pink floyd""",35540
"""the killers""",33081
"""linkin park""",31822


## Graph Data Science (GDS) — present or not, no error either way

Neo4j's **GDS** plugin adds named graph algorithms — PageRank, Louvain, node similarity — that
run over an in-memory *projection* of the graph. GDS is an optional plugin, and **not every
Neo4j carries it** (Community edition does not bundle it). So rather than call a `gds.*`
procedure blind and risk an error output, we **detect** whether GDS is installed first, and only
then run an algorithm — over an **anonymous** native projection, `STREAM` mode, top results only,
so **nothing is persisted**. If GDS is absent, the degree-centrality aggregation above already
gave us the same ranking in plain Cypher, and this cell simply says so.

In [9]:
gds_procs = run("SHOW PROCEDURES YIELD name WHERE name STARTS WITH 'gds.' "
                "RETURN count(name) AS n")[0]["n"]

if gds_procs > 0:
    # GDS 2.x removed anonymous projections: create a NAMED projection, stream, then drop it (transient ->
    # nothing left behind). Project (User)-[:PLAYS]->(Artist) with the edge REVERSED so a node's degree = its
    # incoming plays = listener count -> the GDS equivalent of the native-Cypher degree cell above.
    # The whole path is guarded so a GDS version / edition / memory quirk degrades to a note, not an error.
    G = "nb_degree_tmp"
    try:
        run("CALL gds.graph.drop($g, false) YIELD graphName", g=G)   # idempotent pre-clean
    except Exception:
        pass
    try:
        run("CALL gds.graph.project($g, ['User','Artist'], {PLAYS: {orientation: 'REVERSE'}}) "
            "YIELD nodeCount, relationshipCount", g=G)
        rows = run("""
            CALL gds.degree.stream($g)
            YIELD nodeId, score
            WITH gds.util.asNode(nodeId) AS n, score
            WHERE n:Artist
            RETURN n.name AS artist, toInteger(score) AS listeners
            ORDER BY listeners DESC
            LIMIT 10
        """, g=G)
        print(f"GDS present ({gds_procs} procedures) - gds.degree over a named "
              "(User)-[:PLAYS]->(Artist) projection, REVERSE-oriented so degree = listener count:")
        display(df(rows))
        print("Same ranking as the native-Cypher degree cell above - GDS reproduces it on a projected graph.")
    except Exception as e:
        print(f"GDS present ({gds_procs} procedures) but the degree run was skipped "
              f"({type(e).__name__}: {str(e)[:100]}).")
        print("The native-Cypher degree-centrality aggregation in the previous cell is the equivalent.")
    finally:
        try:
            run("CALL gds.graph.drop($g, false) YIELD graphName", g=G)   # cleanup - nothing left behind
        except Exception:
            pass
else:
    print("GDS is not installed on this Neo4j instance "
          f"({comp['edition']} edition bundles APOC but not GDS).")
    print("No gds.* call is made, so there is no error to leave behind.")
    print("The degree-centrality aggregation in the previous cell is the native-Cypher")
    print("equivalent of gds.degree - same ranking, computed with plain MATCH/count.")

GDS present (423 procedures) - gds.degree over a named (User)-[:PLAYS]->(Artist) projection, REVERSE-oriented so degree = listener count:


artist,listeners
str,i64
"""radiohead""",61984
"""the beatles""",61010
"""coldplay""",53400
"""red hot chili peppers""",39135
"""muse""",37509
"""metallica""",36115
"""pink floyd""",35540
"""the killers""",33081
"""linkin park""",31822


Same ranking as the native-Cypher degree cell above - GDS reproduces it on a projected graph.


## Close the driver

The Neo4j driver holds a connection pool; we close it explicitly, mirroring how notebooks
`20` / `22` release their clients at the end of a session.

In [10]:
driver.close()
print("Neo4j driver closed.")

Neo4j driver closed.


## When to reach for the graph store

**Reach for Neo4j when the value is in the *relationships*, and the question is a traversal:**
- **Multi-hop / recursive patterns** — "things connected to things connected to things": the
  genre-hierarchy rollup and the co-listener walk above. Each is a short Cypher pattern and an
  ugly pile of self-joins in SQL.
- **Co-occurrence and recommendation** — "who also played / bought / cited X" is a two-hop
  pattern anchored on one node; the graph walks only the neighbourhood, not the whole table.
- **Reachability, paths, and centrality** — shortest path, "is A connected to B", who the hubs
  are. Degree centrality is one line; with the GDS plugin, PageRank / community detection are one
  procedure call over a projection.

**Reach for a different store when the shape is not a traversal:**

| you want to… | use |
|--------------|-----|
| walk relationships, multi-hop patterns, paths, centrality | **Neo4j** (this notebook) |
| join heterogeneous engines ad hoc, no ETL | **Trino** federation (notebook `20`) |
| a fast point read/write or aggregation on one specialist store | **native client** (notebook `22`) |
| version / branch / time-travel the data itself | **lakeFS / Nessie+Iceberg** (`10` / `11`) |

The mesh keeps the *same* entities in more than one store on purpose: the artists and plays here
also live as columns in ClickHouse and as rows the marts aggregate. Neo4j earns its copy by
answering the one question the others answer badly — **not "what are the values?" but "what is
connected to what, and how far?"** When that is the question, the relationships being first-class
is the whole game.